# 📊 Data Visualization — Graded Course Project
## BrewLine Café: From Raw Transactions to a Data Story

| | |
|---|---|
| **Course** | Data Visualization |
| **Weight** | Course project (100 points) |
| **Format** | Individual work, one `.ipynb` file |
| **Estimated effort** | 8 – 10 hours |
| **Student name / ID** | *fill in the next cell* |

---

### 🎯 Scenario
**BrewLine** is a fast-growing café chain with **12 stores in 6 Saudi cities**. The management team has handed you
**two years (2023 – 2024) of raw point-of-sale data**. The data was exported straight from the till system, so it is
**messy** — it contains duplicates, missing values, inconsistent spelling and data-entry errors.

Your job is to act as the chain's data analyst: **clean** the data, **describe** it statistically, **visualize** it
with static and interactive charts, **analyze** its time series, **explore** an image dataset used for the chain's
marketing, and finally **communicate** three actionable insights to management.

### 🧮 Grading rubric (100 points)
| Part | Content | Points |
|---|---|---|
| 1 | Data understanding, cleaning & feature engineering | **12** |
| 2 | Descriptive statistics & hypothesis testing | **10** |
| 3 | Visualizing numeric, categorical & ordinal data | **14** |
| 4 | Advanced Matplotlib / Seaborn plotting | **14** |
| 5 | Exploratory data analysis: correlation & PCA | **12** |
| 6 | Time-series visualization, decomposition & forecasting | **14** |
| 7 | Interactive visualization with Plotly | **14** |
| 8 | Data storytelling & reflection | **10** |
| | **Total** | **100** |

**Figure quality criteria (applied inside every plotting task):** correct chart type for the data · title, axis labels with
units · readable fonts · legend only when needed · colour-blind-safe palette · no overplotting / chart junk · sensible
ordering of categories.

**Adjustments:** up to **−5** if the notebook does not run top-to-bottom (*Kernel → Restart & Run All*) or contains
leftover `YOUR CODE HERE` stubs; up to **+3** bonus for outstanding creativity or an extra insightful visualization
(clearly labelled *BONUS*).

### 📜 Rules
1. Use only the libraries listed in the *Setup* cell (all were used in the course labs).
2. **Do not edit** the data-generation cell or cells marked 🔒 *given*.
3. Every plotting task needs a **title, axis labels and readable ticks**. Answers to "interpret" questions must be
   written in the Markdown cell right under the task and must **cite numbers from your output**.
4. Keep the variable names in the *naming contract* below — the instructor's auto-checks rely on them.
5. Work must be your own. Copying code from classmates or generated text without understanding it is academic misconduct.
   You may consult the course notebooks, library documentation and Stack Overflow — cite any non-trivial snippet in a comment.

### 🏷️ Naming contract (used by the 🔒 check cells)
| Variable | Meaning | Defined in |
|---|---|---|
| `tx`, `stores` | raw transactions / stores tables | 1.1 |
| `column_types` | `dict` column → type | 1.2 |
| `df` | **final clean & enriched** transactions DataFrame | 1.3 – 1.6 |
| `rev_stats` | `pd.Series` of revenue statistics | 2.1 |
| `p_ttest`, `p_anova` | p-values (floats) | 2.4 |
| `pca` | `sklearn` PCA fitted with **all** components | 5.2 |
| `daily` | daily revenue `pd.Series` with a daily `DatetimeIndex` | 6.1 |
| `forecast_hw`, `forecast_naive`, `metrics` | 28-day forecasts & error table | 6.5 |

### 📦 Submission
Run *Kernel → Restart & Run All*, make sure no cell errors, then submit **this `.ipynb` file** (with outputs) **and** the
`outputs/` folder it creates (saved figures + HTML). Late policy: as announced by the instructor.

## 0 · Setup  🔒 *(given — run, do not edit)*

In [ ]:
# ✏️ Fill in your details
STUDENT_NAME = "AHMED TURKI ALOTAIBI"
STUDENT_ID = "471110773"

In [ ]:
# 🔒 GIVEN — libraries and global settings
# pip install pandas numpy matplotlib seaborn plotly scipy scikit-learn statsmodels scikit-image opencv-python
import os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display, Markdown

if "google.colab" not in sys.modules:                 # lightweight Plotly renderer for Jupyter / VS Code
    pio.renderers.default = "plotly_mimetype+notebook_connected"

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
os.makedirs("outputs", exist_ok=True)                 # all saved figures must go here

SEED = 2026
WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MONTH_ORDER = ["January", "February", "March", "April", "May", "June", "July",
               "August", "September", "October", "November", "December"]
print("Setup OK — pandas", pd.__version__, "| seaborn", sns.__version__)

### 🔒 Data generation *(given — do not edit)*
The function below simulates the BrewLine point-of-sale export and writes two CSV files to `data/`.
Prices are in **Saudi riyals (SAR)**. Weekend in Saudi Arabia = **Friday & Saturday**.

| File | Columns |
|---|---|
| `brewline_transactions.csv` | `transaction_id, date, store_id, city, category, product, quantity, unit_price, discount, customer_segment, payment_method, customer_age, prep_time_min, rating` |
| `brewline_stores.csv` | `store_id, city, lat, lon, store_size` |

`rating` is the customer satisfaction score from **1 (poor) to 5 (excellent)** and `prep_time_min` the minutes needed to prepare the order.

In [ ]:
# 🔒 GIVEN — synthetic, reproducible data generator (do not edit)
def make_brewline_data(seed=2026, n_rows=12000, out_dir="data"):
    rng = np.random.default_rng(seed)
    os.makedirs(out_dir, exist_ok=True)

    # ---------- stores ----------
    city_ll = {"Riyadh": (24.7136, 46.6753), "Jeddah": (21.4858, 39.1925), "Dammam": (26.4207, 50.0888),
               "Mecca": (21.3891, 39.8579), "Medina": (24.5247, 39.5692), "Abha": (18.2164, 42.5053)}
    store_cities = ["Riyadh"] * 4 + ["Jeddah"] * 3 + ["Dammam"] * 2 + ["Mecca", "Medina", "Abha"]
    stores = pd.DataFrame({"store_id": [f"S{i:02d}" for i in range(1, 13)], "city": store_cities})
    stores["lat"] = stores.city.map(lambda c: city_ll[c][0]) + rng.normal(0, 0.04, 12)
    stores["lon"] = stores.city.map(lambda c: city_ll[c][1]) + rng.normal(0, 0.04, 12)
    stores["store_size"] = rng.choice(["Small", "Medium", "Large"], 12, p=[.3, .45, .25])
    size_w = stores.store_size.map({"Small": .6, "Medium": 1.0, "Large": 1.7}).to_numpy()
    city_w = stores.city.map({"Riyadh": 1.3, "Jeddah": 1.1, "Dammam": .9, "Mecca": .8, "Medina": .7, "Abha": .5}).to_numpy()
    sp = size_w * city_w; sp = sp / sp.sum()

    # ---------- dates: growth trend + weekend peak + yearly seasonality ----------
    dates = pd.date_range("2023-01-01", "2024-12-31", freq="D")
    t = np.arange(len(dates)); dow = dates.dayofweek.to_numpy()
    w = (1 + 0.0010 * t) * np.where(np.isin(dow, [4, 5]), 1.35, 1.0) \
        * (1 + 0.12 * np.sin(2 * np.pi * (dates.dayofyear.to_numpy() - 30) / 365.25))
    date = pd.to_datetime(rng.choice(dates.to_numpy(), size=n_rows, p=w / w.sum()))
    month, dow_r = date.month.to_numpy(), date.dayofweek.to_numpy()
    store_idx = rng.choice(12, size=n_rows, p=sp)

    # ---------- category / product (cold drinks boom in summer) ----------
    cats = ["Coffee", "Tea", "Cold Drinks", "Bakery", "Merchandise"]
    catalog = {"Coffee": [("Espresso", 12), ("Latte", 18), ("Cappuccino", 17), ("Flat White", 19), ("Arabic Coffee", 10)],
               "Tea": [("Karak Tea", 9), ("Green Tea", 11), ("Mint Tea", 10)],
               "Cold Drinks": [("Iced Latte", 20), ("Cold Brew", 22), ("Lemonade", 14), ("Iced Tea", 13)],
               "Bakery": [("Croissant", 10), ("Date Maamoul", 8), ("Cheesecake", 24), ("Muffin", 11)],
               "Merchandise": [("Tumbler", 55), ("Coffee Beans 250g", 48), ("Mug", 35)]}
    p_winter = np.cumsum([.40, .15, .15, .24, .06]); p_summer = np.cumsum([.30, .10, .38, .17, .05])
    u = rng.random(n_rows); summer = np.isin(month, [6, 7, 8, 9])
    cat_idx = np.minimum(np.where(summer, np.searchsorted(p_summer, u), np.searchsorted(p_winter, u)), 4)
    product = np.empty(n_rows, dtype=object); base = np.zeros(n_rows)
    for ci, c in enumerate(cats):
        idx = np.where(cat_idx == ci)[0]; items = catalog[c]
        pick = rng.integers(0, len(items), len(idx))
        product[idx] = [items[k][0] for k in pick]; base[idx] = [items[k][1] for k in pick]

    # ---------- customers, prices, quantities ----------
    seg = rng.choice(["Regular", "New", "Loyalty", "Corporate"], n_rows, p=[.45, .25, .22, .08])
    lam = np.where(np.isin(dow_r, [4, 5]), 0.75, 0.40) + np.where(seg == "Corporate", 1.8, 0.0)
    qty = 1 + rng.poisson(lam); qty = np.where(cat_idx == 4, np.minimum(qty, 3), qty)
    price = np.round(base * np.where(date.year == 2024, 1.04, 1.0) * rng.normal(1, 0.03, n_rows), 2)
    disc = np.zeros(n_rows)
    for s, (vals, pr) in {"Loyalty": ([0, .05, .10], [.3, .4, .3]), "New": ([0, .10, .20], [.5, .3, .2]),
                          "Regular": ([0, .05], [.8, .2]), "Corporate": ([0, .05], [.8, .2])}.items():
        m = seg == s; disc[m] = rng.choice(vals, m.sum(), p=pr)
    pay = rng.choice(["Card", "Cash", "Mobile Wallet", "Apple Pay"], n_rows, p=[.40, .15, .25, .20])
    age = np.clip(rng.normal(np.where(seg == "Corporate", 40, 31), 9), 16, 70).round()
    prep = np.clip(2 + 1.1 * qty + np.where(cat_idx == 3, -1.0, 0.0) + np.where(np.isin(dow_r, [4, 5]), .6, 0.0)
                   + rng.normal(0, .8, n_rows), 1, None).round(1)
    rating = np.clip(np.round(4.7 - 0.12 * prep + rng.normal(0, .7, n_rows)), 1, 5)

    df = pd.DataFrame({"date": date, "store_id": stores.store_id.to_numpy()[store_idx],
                       "city": stores.city.to_numpy()[store_idx], "category": np.array(cats)[cat_idx],
                       "product": product, "quantity": qty, "unit_price": price, "discount": disc,
                       "customer_segment": seg, "payment_method": pay, "customer_age": age,
                       "prep_time_min": prep, "rating": rating})
    df = df.sort_values("date").reset_index(drop=True)
    df.insert(0, "transaction_id", [f"T{i:06d}" for i in range(1, len(df) + 1)])
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    # ---------- make it messy ----------
    for col, frac in [("customer_age", .08), ("rating", .12), ("payment_method", .03)]:
        df.loc[rng.random(len(df)) < frac, col] = np.nan
    m = rng.random(len(df)) < .06; df.loc[m, "city"] = df.loc[m, "city"].str.upper()
    m = rng.random(len(df)) < .05; df.loc[m, "city"] = " " + df.loc[m, "city"].str.lower() + " "
    m = rng.random(len(df)) < .04; df.loc[m, "category"] = df.loc[m, "category"].str.lower() + " "
    o = rng.choice(len(df), 10, replace=False); df.loc[o, "unit_price"] = (df.loc[o, "unit_price"] * 10).round(2)
    o = rng.choice(len(df), 6, replace=False);  df.loc[o, "quantity"] = rng.integers(100, 300, 6)
    df = pd.concat([df, df.sample(80, random_state=seed)]).sample(frac=1, random_state=seed).reset_index(drop=True)

    df.to_csv(f"{out_dir}/brewline_transactions.csv", index=False)
    stores.round(5).to_csv(f"{out_dir}/brewline_stores.csv", index=False)
    return df.shape, stores.shape

print(make_brewline_data(seed=SEED))

---
# Part 1 · Data Understanding, Cleaning & Feature Engineering  *(12 pts)*
*Course topics: data types, data scrubbing, data transformation.*

### Task 1.1 — Load and inspect  *(2 pts)*
1. Load both CSV files into `tx` and `stores` (`pd.read_csv`).
2. Show the shape, the first rows, `.info()` and `.describe(include="all")` of `tx`.
3. Build a DataFrame `missing` with the **count and percentage** of missing values per column (only columns with missing values).
4. Print the number of fully duplicated rows.

In [ ]:

import pandas as pd
from tkinter import Tk, filedialog

root = Tk()
root.withdraw()

file_path = filedialog.askopenfilename(
    title="Select Transactions Excel File",
    filetypes=[("Excel files", "*.xlsx *.xls")]
)

if file_path:
    tx = pd.read_excel(file_path)
    print(tx.head())
else:
    print("No file selected")

root.destroy()


In [ ]:
import os

print(os.listdir("data"))

In [ ]:
import pandas as pd

# Load CSV files
tx = pd.read_csv("data/brewline_transactions.csv")
stores = pd.read_csv("data/brewline_stores.csv")

# Display first 5 rows
print("Transactions:")
display(tx.head())

print("Stores:")
display(stores.head())

In [ ]:
# Check dataset information
print("Transactions shape:", tx.shape)
print("Stores shape:", stores.shape)

# Check column types
tx.info()

# Check missing values
print("\nMissing values:")
print(tx.isnull().sum())

### Task 1.2 — Classify the variables  *(1 pt)*
Create a dictionary `column_types` that maps **every column of `tx`** to one of
`"identifier"`, `"datetime"`, `"nominal"`, `"ordinal"`, `"discrete"`, `"continuous"`.
Add a one-line comment for any column you found debatable.

In [ ]:
df = tx.copy()
column_types = {
    "transaction_id": "identifier",
    "date": "datetime",
    "store_id": "identifier",
    "city": "nominal",
    "category": "nominal",
    "product": "nominal",
    "quantity": "discrete",
    "unit_price": "continuous",
    "discount": "continuous",
    "customer_segment": "nominal",
    "payment_method": "nominal",
    "customer_age": "continuous",  # Age is continuous, although recorded in years.
    "prep_time_min": "continuous",
    "rating": "ordinal"  # Ratings are ordered categories from 1 to 5.
}

print(column_types)

### Task 1.3 — Standardise formats and remove duplicates  *(2 pts)*
Work on a **copy**: `df = tx.copy()`. Then
* convert `date` to `datetime64`;
* strip white-space and fix the capitalisation of `city` and `category` (e.g. `" riyadh "`, `"RIYADH"` → `"Riyadh"`; `"coffee "` → `"Coffee"`) and strip `product`, `customer_segment`, `payment_method`;
* print `df["city"].unique()` and `df["category"].unique()` to prove the fix;
* drop duplicated transactions (same `transaction_id`) and report how many rows were removed.

In [ ]:
# Work on a copy
df = tx.copy()

# Convert date to datetime
df["date"] = pd.to_datetime(df["date"])

# Standardise city and category
df["city"] = df["city"].str.strip().str.title()
df["category"] = df["category"].str.strip().str.title()

# Strip whitespace from other columns
for col in ["product", "customer_segment", "payment_method"]:
    df[col] = df[col].str.strip()

# Check unique values
print("Unique cities:", df["city"].unique())
print("Unique categories:", df["category"].unique())

# Remove duplicate transactions
rows_before = len(df)

df = df.drop_duplicates(subset="transaction_id", keep="first")

rows_after = len(df)
rows_removed = rows_before - rows_after

print("Rows removed:", rows_removed)
print("Rows remaining:", rows_after)

### Task 1.4 — Handle missing values  *(2 pts)*
Choose and **justify (in a code comment)** a strategy per column:
* `customer_age` → impute with the **median age of the same `customer_segment`**;
* `payment_method` → fill with the label `"Unknown"` (do not guess a payment method);
* `rating` → impute with the **median rating of the same `category`** and convert to integer.

Afterwards, print the remaining number of missing values (must be 0 for these three columns).

In [ ]:
# Fill missing customer_age with the median age of the same customer_segment
# This preserves age differences between customer segments.
df["customer_age"] = df["customer_age"].fillna(
    df.groupby("customer_segment")["customer_age"].transform("median")
)

# Fill missing payment_method with "Unknown"
# This avoids guessing the customer's payment method.
df["payment_method"] = df["payment_method"].fillna("Unknown")

# Fill missing rating with the median rating of the same category
# This preserves category-specific rating patterns.
df["rating"] = df["rating"].fillna(
    df.groupby("category")["rating"].transform("median")
)

# Convert rating to integer
df["rating"] = df["rating"].round().astype(int)

# Check remaining missing values
print("Remaining missing values:")
print(df[["customer_age", "payment_method", "rating"]].isnull().sum())

### Task 1.5 — Detect and treat outliers  *(2 pts)*
Some prices were typed with an extra zero and some quantities contain typing errors.
1. **`unit_price`**: for each `product` flag values outside `[Q1 − 3·IQR, Q3 + 3·IQR]` (extreme-outlier rule, computed **per product**).
2. **`quantity`**: flag values `> 20` (domain rule — nobody buys 100 coffees at one till).
3. Show a box plot (or strip plot) of `unit_price` for the flagged products **before** removal, print how many rows are flagged, then **remove** them and reset the index.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate Q1, Q3, and IQR for each product
Q1 = df.groupby("product")["unit_price"].transform(
    lambda x: x.quantile(0.25)
)

Q3 = df.groupby("product")["unit_price"].transform(
    lambda x: x.quantile(0.75)
)

IQR = Q3 - Q1

# Flag extreme price outliers per product
price_outliers = (
    (df["unit_price"] < Q1 - 3 * IQR) |
    (df["unit_price"] > Q3 + 3 * IQR)
)

# Flag quantity outliers greater than 20
quantity_outliers = df["quantity"] > 20

# Combine both outlier conditions
flagged = price_outliers | quantity_outliers

# Show box plot before removal
flagged_products = df.loc[price_outliers, "product"].unique()

if len(flagged_products) > 0:
    plt.figure(figsize=(12, 6))
    sns.boxplot(
        data=df[df["product"].isin(flagged_products)],
        x="product",
        y="unit_price"
    )
    plt.xticks(rotation=45, ha="right")
    plt.title("Unit Price Outliers Before Removal")
    plt.tight_layout()
    plt.show()

# Print number of flagged rows
print("Price outliers:", price_outliers.sum())
print("Quantity outliers:", quantity_outliers.sum())
print("Total flagged rows:", flagged.sum())

# Remove flagged rows and reset index
df = df.loc[~flagged].reset_index(drop=True)

print("Rows remaining:", len(df))

### Task 1.6 — Feature engineering and merge  *(3 pts)*
Add the following columns to `df`:

| New column | Definition |
|---|---|
| `revenue` | `quantity × unit_price × (1 − discount)`, rounded to 2 decimals |
| `year`, `month` (1–12), `month_name`, `year_month` (text like `"2023-04"`), `day_of_week` (name) | from `date` |
| `is_weekend` | `True` for **Friday and Saturday** |
| `age_group` | `pd.cut` of `customer_age` into `"<25"`, `"25-34"`, `"35-44"`, `"45+"` |

Then **merge** `store_size`, `lat`, `lon` from `stores` on `store_id` (left join), and save the clean table to `outputs/brewline_clean.csv`.

In [ ]:
import pandas as pd
import os

# Calculate revenue
df["revenue"] = (
    df["quantity"] * df["unit_price"] * (1 - df["discount"])
).round(2)

# Extract date features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["year_month"] = df["date"].dt.strftime("%Y-%m")
df["day_of_week"] = df["date"].dt.day_name()

# Friday and Saturday are weekends
df["is_weekend"] = df["date"].dt.dayofweek.isin([4, 5])

# Create age groups
df["age_group"] = pd.cut(
    df["customer_age"],
    bins=[-float("inf"), 25, 35, 45, float("inf")],
    labels=["<25", "25-34", "35-44", "45+"],
    right=False
)

# Merge store information
df = df.merge(
    stores[["store_id", "store_size", "lat", "lon"]],
    on="store_id",
    how="left",
    validate="many_to_one"
)

# Create outputs folder
os.makedirs("outputs", exist_ok=True)

# Save cleaned dataset
df.to_csv("outputs/brewline_clean.csv", index=False)

# Display results
print("Clean dataset shape:", df.shape)
print("File saved to: outputs/brewline_clean.csv")

display(df.head())

In [ ]:
# 🔒 CHECK — Part 1 (given)
assert df["transaction_id"].is_unique, "duplicates still present"
assert df[["customer_age", "rating", "payment_method"]].isna().sum().sum() == 0, "missing values remain"
assert {"revenue", "is_weekend", "year_month", "day_of_week", "age_group", "lat", "lon", "store_size"} <= set(df.columns), "missing engineered columns"
assert df["quantity"].max() <= 20 and df["unit_price"].max() < 100, "outliers remain"
assert set(df["category"].unique()) == {"Coffee", "Tea", "Cold Drinks", "Bakery", "Merchandise"}, "category labels not standardised"
assert set(df["city"].unique()) == {"Riyadh", "Jeddah", "Dammam", "Mecca", "Medina", "Abha"}, "city labels not standardised"
assert set(column_types) == set(tx.columns), "column_types must cover every column of tx"
print("✅ Part 1 checks passed —", len(df), "clean rows")

---
# Part 2 · Descriptive Statistics & Hypothesis Testing  *(10 pts)*
*Course topic 3: centrality, dispersion, distributions, p-values.*

### Task 2.1 — Summary statistics of `revenue`  *(2 pts)*
Create the `pd.Series` **`rev_stats`** (rounded to 2 decimals) with the index
`mean, median, mode, range, IQR, variance, std, skewness, kurtosis` (compute the mode on revenue rounded to whole riyals).

In [ ]:
rev_stats = None
rev = df["revenue"]

rev_stats = pd.Series({
    "mean": rev.mean(),
    "median": rev.median(),
    "mode": rev.round(0).mode().iloc[0],
    "range": rev.max() - rev.min(),
    "IQR": rev.quantile(0.75) - rev.quantile(0.25),
    "variance": rev.var(),
    "std": rev.std(),
    "skewness": rev.skew(),
    "kurtosis": rev.kurt()
}).round(2)

print(rev_stats)

### Task 2.2 — Group statistics  *(2 pts)*
Build `by_category`: for each `category` the **count, mean, median, std and total (sum)** of `revenue`, sorted by total revenue (descending), rounded to 2 decimals. Display it.

In [ ]:
by_category = (
    df.groupby("category")["revenue"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        total="sum"
    )
    .sort_values(by="total", ascending=False)
    .round(2)
)

display(by_category)

### Task 2.3 — Distribution of revenue  *(3 pts)*
Draw **one figure with two side-by-side panels**:
* left: histogram + KDE of `revenue` with **vertical lines for the mean and the median** (labelled in the legend);
* right: the same for `log10(revenue)`.

Then answer: *Is the raw distribution symmetric? What do mean vs. median and the skewness value tell you? Why is the log scale helpful?*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

revenue = df["revenue"].dropna()
log_revenue = np.log10(revenue[revenue > 0])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Raw revenue distribution
sns.histplot(revenue, bins=40, kde=True, ax=axes[0])

axes[0].axvline(revenue.mean(), color="red", linestyle="--",
                label=f"Mean: {revenue.mean():.2f}")
axes[0].axvline(revenue.median(), color="green", linestyle="--",
                label=f"Median: {revenue.median():.2f}")

axes[0].set_title("Revenue Distribution")
axes[0].set_xlabel("Revenue (SAR)")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# Right: Log10 revenue distribution
sns.histplot(log_revenue, bins=40, kde=True, ax=axes[1])

axes[1].axvline(log_revenue.mean(), color="red", linestyle="--",
                label=f"Mean: {log_revenue.mean():.2f}")
axes[1].axvline(log_revenue.median(), color="green", linestyle="--",
                label=f"Median: {log_revenue.median():.2f}")

axes[1].set_title("Log10 Revenue Distribution")
axes[1].set_xlabel("Log10(Revenue)")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.show()

# Interpretation
skewness = revenue.skew()

print(f"Revenue skewness: {skewness:.2f}")
print(f"Mean revenue: {revenue.mean():.2f}")
print(f"Median revenue: {revenue.median():.2f}")

if skewness > 0.5:
    print("The raw revenue distribution is not symmetric; it is right-skewed.")
    print("The mean is pulled upward by high-revenue transactions.")
elif skewness < -0.5:
    print("The raw revenue distribution is not symmetric; it is left-skewed.")
    print("Low-revenue transactions pull the mean downward.")
else:
    print("The raw revenue distribution is approximately symmetric.")

print("The log10 scale compresses large values, reduces the effect of extreme values,")
print("and makes the distribution easier to visualize and compare.")

✍️ **Your answer:** *(The raw revenue distribution is not symmetric; it is right-skewed. The mean is higher than the median, and the positive skewness indicates that some high-revenue transactions pull the mean upward. The log10 scale compresses larger values, reduces the skewness, and makes the distribution easier to visualize.)*

### Task 2.4 — Hypothesis tests  *(3 pts)*
Use `scipy.stats` (significance level α = 0.05).
1. **Welch two-sample t-test**: is the mean `revenue` of **weekend** transactions different from **weekday** transactions? Store the p-value in **`p_ttest`**.
2. **One-way ANOVA**: does mean `revenue` differ across the four `customer_segment`s? Store the p-value in **`p_anova`**.
3. State H₀/H₁, the decision, and a business interpretation for each test. *(Bonus thought: revenue is skewed — would a non-parametric test change the conclusion?)*

In [ ]:
p_ttest = None
p_anova = None
from scipy import stats

alpha = 0.05

# 1. Welch two-sample t-test
weekend = df.loc[df["is_weekend"], "revenue"].dropna()
weekday = df.loc[~df["is_weekend"], "revenue"].dropna()

t_stat, p_ttest = stats.ttest_ind(
    weekend, weekday, equal_var=False
)

print("Welch t-test")
print(f"p-value: {p_ttest:.6f}")
print("H0: Mean revenue is equal for weekend and weekday transactions.")
print("H1: Mean revenue is different between weekend and weekday transactions.")

if p_ttest < alpha:
    print("Decision: Reject H0.")
    print("Business interpretation: Weekend and weekday transactions have significantly different mean revenue.")
else:
    print("Decision: Fail to reject H0.")
    print("Business interpretation: There is insufficient evidence of a difference in mean revenue.")

print()

# 2. One-way ANOVA
segments = [
    group["revenue"].dropna()
    for _, group in df.groupby("customer_segment")
]

f_stat, p_anova = stats.f_oneway(*segments)

print("One-way ANOVA")
print(f"p-value: {p_anova:.6f}")
print("H0: All customer segments have equal mean revenue.")
print("H1: At least one customer segment has a different mean revenue.")

if p_anova < alpha:
    print("Decision: Reject H0.")
    print("Business interpretation: Mean revenue differs significantly across customer segments.")
else:
    print("Decision: Fail to reject H0.")
    print("Business interpretation: There is insufficient evidence of differences in mean revenue across customer segments.")

# Bonus: Non-parametric tests
u_stat, p_mannwhitney = stats.mannwhitneyu(
    weekend, weekday, alternative="two-sided"
)

h_stat, p_kruskal = stats.kruskal(*segments)

print("\nBonus: Non-parametric tests")
print(f"Mann-Whitney U p-value: {p_mannwhitney:.6f}")
print(f"Kruskal-Wallis p-value: {p_kruskal:.6f}")

print("Weekend conclusion changed:",
      (p_ttest < alpha) != (p_mannwhitney < alpha))
print("Segment conclusion changed:",
      (p_anova < alpha) != (p_kruskal < alpha))

✍️ **Your answer:** *(For the Welch t-test, H₀ states that weekend and weekday mean revenues are equal, while H₁ states they differ. Since p < 0.05, we reject H₀, indicating a statistically significant difference that may help the business plan its weekend operations. For ANOVA, H₀ states that all customer segments have equal mean revenue, while H₁ states that at least one differs; since p < 0.05, we reject H₀, suggesting that customer segmentation may support targeted marketing strategies. The non-parametric tests confirm the same conclusions despite the skewed revenue distribution.)*

In [ ]:
# 🔒 CHECK — Part 2 (given)
assert isinstance(rev_stats, pd.Series) and {"mean", "median", "mode", "range", "IQR", "variance", "std", "skewness", "kurtosis"} <= set(rev_stats.index)
assert 0 <= float(p_ttest) <= 1 and 0 <= float(p_anova) <= 1
assert abs(rev_stats["mean"] - df["revenue"].mean()) < 0.01
print("✅ Part 2 checks passed")

---
# Part 3 · Visualizing Numeric, Categorical & Ordinal Data  *(14 pts)*
*Course topic 2: histograms, box/violin plots, scatter & line plots, bar/count plots, pie charts, colour theory.*

Use a **colour-blind-safe palette** throughout (e.g. `sns.color_palette("colorblind")`).

### Task 3.1 — Four numeric views in one figure  *(4 pts)*
Create a `2 × 2` grid (`plt.subplots`) with:
1. **Histogram** of `customer_age` (choose a sensible number of bins);
2. **Box plot** of `unit_price` for each `category` (ordered by number of transactions);
3. **Scatter plot** of `prep_time_min` (x) vs `revenue` (y) on a **random sample of 1 500 rows**, coloured by `category` (use transparency to fight over-plotting);
4. **Line plot** of total monthly revenue (`year_month`) with markers and rotated tick labels.

Every panel needs its own title and axis labels; add one overall `suptitle`.

In [ ]:
PAL = sns.color_palette("colorblind")
import matplotlib.pyplot as plt
import seaborn as sns

# Colour-blind-safe palette
sns.set_palette("colorblind")
palette = sns.color_palette("colorblind")

# Create 2x2 figure
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Histogram of customer age
sns.histplot(
    data=df,
    x="customer_age",
    bins=20,
    color=palette[0],
    ax=axes[0, 0]
)
axes[0, 0].set_title("Distribution of Customer Age")
axes[0, 0].set_xlabel("Customer Age (Years)")
axes[0, 0].set_ylabel("Frequency")

# 2. Box plot of unit price by category
category_order = df["category"].value_counts().index

sns.boxplot(
    data=df,
    x="category",
    y="unit_price",
    order=category_order,
    hue="category",
    palette="colorblind",
    legend=False,
    ax=axes[0, 1]
)
axes[0, 1].set_title("Unit Price by Category")
axes[0, 1].set_xlabel("Category")
axes[0, 1].set_ylabel("Unit Price (SAR)")
axes[0, 1].tick_params(axis="x", rotation=30)

# 3. Scatter plot using random sample of 1500 rows
sample_df = df.sample(n=1500, random_state=42)

sns.scatterplot(
    data=sample_df,
    x="prep_time_min",
    y="revenue",
    hue="category",
    palette="colorblind",
    alpha=0.5,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Preparation Time vs Revenue")
axes[1, 0].set_xlabel("Preparation Time (Minutes)")
axes[1, 0].set_ylabel("Revenue (SAR)")
axes[1, 0].legend(title="Category")

# 4. Line plot of monthly total revenue
monthly_revenue = (
    df.groupby("year_month")["revenue"]
    .sum()
    .sort_index()
)

sns.lineplot(
    x=monthly_revenue.index,
    y=monthly_revenue.values,
    marker="o",
    color=palette[0],
    ax=axes[1, 1]
)
axes[1, 1].set_title("Total Monthly Revenue")
axes[1, 1].set_xlabel("Year-Month")
axes[1, 1].set_ylabel("Total Revenue (SAR)")
axes[1, 1].tick_params(axis="x", rotation=45)

# Overall title
fig.suptitle(
    "Brewline Coffee - Numeric Data Visualizations",
    fontsize=18,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### Task 3.2 — Three categorical views  *(4 pts)*
In a `1 × 3` figure (`figsize≈(18, 5)`) draw:
1. a **count plot** of `category` (horizontal bars, sorted);
2. a **bar chart of mean `revenue` per `payment_method`** with 95 % confidence-interval error bars, bars sorted;
3. a **donut chart** (pie with a hole) of the share of transactions per `customer_segment` with percentage labels.

Store the figure in the variable `fig_cat`.

In [ ]:
fig_cat = None
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_palette("colorblind")
palette = sns.color_palette("colorblind")

fig_cat, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Horizontal count plot of category
category_order = df["category"].value_counts().index

sns.countplot(
    data=df,
    y="category",
    order=category_order,
    hue="category",
    palette="colorblind",
    legend=False,
    ax=axes[0]
)

axes[0].set_title("Number of Transactions by Category")
axes[0].set_xlabel("Transaction Count")
axes[0].set_ylabel("Category")

# 2. Mean revenue by payment method with 95% CI
payment_order = (
    df.groupby("payment_method")["revenue"]
    .mean()
    .sort_values(ascending=False)
    .index
)

sns.barplot(
    data=df,
    x="payment_method",
    y="revenue",
    order=payment_order,
    hue="payment_method",
    palette="colorblind",
    errorbar=("ci", 95),
    capsize=0.15,
    legend=False,
    ax=axes[1]
)

axes[1].set_title("Mean Revenue by Payment Method (95% CI)")
axes[1].set_xlabel("Payment Method")
axes[1].set_ylabel("Mean Revenue (SAR)")
axes[1].tick_params(axis="x", rotation=45)

# 3. Donut chart of customer segments
segment_counts = df["customer_segment"].value_counts()

axes[2].pie(
    segment_counts.values,
    labels=segment_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=palette[:len(segment_counts)],
    wedgeprops={"width": 0.4, "edgecolor": "white"}
)

axes[2].set_title("Transaction Share by Customer Segment")
axes[2].axis("equal")

fig_cat.suptitle(
    "Brewline Coffee - Categorical Data Visualizations",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

### Task 3.3 — Compare distributions with a violin plot  *(2 pts)*
Draw a **violin plot** of `revenue` for each `customer_segment`, using a **log-scaled y-axis** (`log_scale=True` or `plt.yscale("log")`),
segments **ordered by median revenue**, with the inner box visible. Add a title that states the finding.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Order customer segments by median revenue
segment_order = (
    df.groupby("customer_segment")["revenue"]
    .median()
    .sort_values(ascending=False)
    .index
)

# Create violin plot
plt.figure(figsize=(10, 6))

sns.violinplot(
    data=df[df["revenue"] > 0],
    x="customer_segment",
    y="revenue",
    order=segment_order,
    hue="customer_segment",
    palette="colorblind",
    inner="box",
    legend=False
)

# Log-scaled y-axis
plt.yscale("log")

# Title based on the highest median revenue segment
top_segment = segment_order[0]

plt.title(
    f"{top_segment} Customers Have the Highest Median Revenue",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Customer Segment")
plt.ylabel("Revenue (SAR) - Log Scale")

plt.tight_layout()
plt.show()

### Task 3.4 — Ordinal data: satisfaction ratings  *(2 pts)*
`rating` is **ordinal (1–5)**. Create a **100 % stacked horizontal bar chart** showing the percentage of each rating within every `category`
(`pd.crosstab(..., normalize="index")`). Because the data are ordinal use an **ordered, sequential colour map** (not a qualitative palette), place the legend outside the plot and label the x-axis in %.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Calculate percentage of each rating within each category
rating_pct = pd.crosstab(
    df["category"],
    df["rating"],
    normalize="index"
) * 100

# Ensure ratings are ordered from 1 to 5
rating_pct = rating_pct.reindex(columns=[1, 2, 3, 4, 5], fill_value=0)

# Use an ordered sequential colour map
colors = plt.cm.Blues(np.linspace(0.25, 0.95, 5))

# Create 100% stacked horizontal bar chart
fig, ax = plt.subplots(figsize=(11, 6))

rating_pct.plot(
    kind="barh",
    stacked=True,
    color=colors,
    width=0.7,
    ax=ax
)

ax.set_title(
    "Satisfaction Rating Distribution by Category",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Percentage of Transactions (%)")
ax.set_ylabel("Category")
ax.set_xlim(0, 100)

# Place legend outside the plot
ax.legend(
    title="Rating",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

### Task 3.5 — Save and justify  *(2 pts)*
1. Save `fig_cat` as **`outputs/fig_3_categorical.png`** at **300 dpi** with a tight bounding box.
2. In 2–3 sentences explain **why your palette is accessible** (colour-vision deficiency, print in grey-scale) and why you used a sequential map for Task 3.4 but a qualitative one elsewhere.

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

fig_cat.savefig(
    "outputs/fig_3_categorical.png",
    dpi=300,
    bbox_inches="tight"
)

print("Figure saved successfully!")

✍️ **Your answer:** *(The colorblind-safe palette makes the charts more accessible to people with colour-vision deficiencies, while clear labels and contrasting shades improve readability in grey-scale printing. A sequential colour map was used for satisfaction ratings because they are ordinal and have a natural order from 1 to 5. Qualitative colours were used for other categorical variables because they represent distinct groups without a natural order.)*

---
# Part 4 · Advanced Plotting with Matplotlib & Seaborn  *(14 pts)*
*Course topic 3 (advanced): subplots/GridSpec, heat-maps, colour maps & colour bars, customisation, combining Matplotlib + Seaborn.*

### Task 4.1 — A GridSpec dashboard  *(4 pts)*
Using `matplotlib.gridspec.GridSpec(2, 3)` build **one figure with three panels of different sizes**:
* **top-left** (1 cell): total revenue per `city`, horizontal bars sorted descending;
* **top-middle + top-right** (2 cells): **average daily revenue per weekday** (first sum revenue per date, then average per weekday, ordered Monday → Sunday). Colour **weekend days differently** from weekdays;
* **bottom row** (all 3 cells): **monthly revenue stacked by `category`** (stacked bars, one bar per `year_month`).

Add a bold `suptitle`, consistent fonts, and make sure nothing overlaps (`tight_layout`/`constrained_layout`).

In [ ]:
from matplotlib.gridspec import GridSpec
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Colour-blind-safe palette
sns.set_palette("colorblind")
colors = sns.color_palette("colorblind")
plt.rcParams.update({"font.size": 11})

# Create GridSpec dashboard
fig = plt.figure(figsize=(18, 12), constrained_layout=True)
gs = GridSpec(2, 3, figure=fig, height_ratios=[1, 1.3])

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1:3])
ax3 = fig.add_subplot(gs[1, :])

# 1. Total revenue by city
city_revenue = (
    df.groupby("city")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

sns.barplot(
    x=city_revenue.values,
    y=city_revenue.index,
    color=colors[0],
    ax=ax1
)

ax1.set_title("Total Revenue by City", fontweight="bold")
ax1.set_xlabel("Total Revenue (SAR)")
ax1.set_ylabel("City")
ax1.invert_yaxis()

# 2. Average daily revenue by weekday
daily_revenue = (
    df.groupby("date")["revenue"]
    .sum()
    .reset_index()
)

daily_revenue["day_of_week"] = daily_revenue["date"].dt.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

weekday_avg = (
    daily_revenue.groupby("day_of_week")["revenue"]
    .mean()
    .reindex(weekday_order)
)

weekday_colors = [
    colors[1] if day in ["Friday", "Saturday"] else colors[0]
    for day in weekday_order
]

ax2.bar(
    weekday_avg.index,
    weekday_avg.values,
    color=weekday_colors
)

ax2.set_title("Average Daily Revenue by Weekday", fontweight="bold")
ax2.set_xlabel("Day of Week")
ax2.set_ylabel("Average Daily Revenue (SAR)")
ax2.tick_params(axis="x", rotation=30)

from matplotlib.patches import Patch

ax2.legend(
    handles=[
        Patch(color=colors[0], label="Weekday"),
        Patch(color=colors[1], label="Weekend")
    ]
)

# 3. Monthly revenue stacked by category
monthly_category = (
    df.groupby(["year_month", "category"])["revenue"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

monthly_category.plot(
    kind="bar",
    stacked=True,
    ax=ax3,
    color=colors[:len(monthly_category.columns)],
    width=0.8
)

ax3.set_title("Monthly Revenue by Category", fontweight="bold")
ax3.set_xlabel("Year-Month")
ax3.set_ylabel("Total Revenue (SAR)")
ax3.tick_params(axis="x", rotation=45)

ax3.legend(
    title="Category",
    bbox_to_anchor=(1.01, 1),
    loc="upper left"
)

# Overall dashboard title
fig.suptitle(
    "Brewline Coffee - Revenue Performance Dashboard",
    fontsize=20,
    fontweight="bold"
)

plt.show()

### Task 4.2 — Heat-map: when does each category sell?  *(3 pts)*
Create a pivot table of **total revenue (SAR thousand)** with `month` (rows, labelled Jan…Dec) × `category` (columns) and draw an **annotated Seaborn heat-map**
with a **sequential colour map**, a **labelled colour bar** (`cbar_kws`), thin cell lines and a descriptive title.
Then mention in one sentence the clearest seasonal pattern.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import calendar

# Create pivot table: total revenue in SAR thousands
monthly_pivot = pd.pivot_table(
    df,
    values="revenue",
    index="month",
    columns="category",
    aggfunc="sum",
    fill_value=0
) / 1000

# Order months from January to December
monthly_pivot = monthly_pivot.reindex(range(1, 13), fill_value=0)
monthly_pivot.index = [calendar.month_abbr[m] for m in range(1, 13)]

# Draw annotated heat-map
fig, ax = plt.subplots(figsize=(12, 8))

sns.heatmap(
    monthly_pivot,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Total Revenue (SAR thousands)"},
    ax=ax
)

ax.set_title(
    "Monthly Revenue by Product Category: Seasonal Sales Patterns",
    fontsize=15,
    fontweight="bold",
    pad=15
)

ax.set_xlabel("Product Category")
ax.set_ylabel("Month")
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

# Identify the clearest seasonal pattern
peak_month = monthly_pivot.sum(axis=1).idxmax()
low_month = monthly_pivot.sum(axis=1).idxmin()

print(
    f"Seasonal pattern: Total revenue peaks in {peak_month} "
    f"and reaches its lowest level in {low_month}."
)

✍️ **Your answer:** *(The heat-map shows how total revenue varies across product categories throughout the year. The clearest seasonal pattern is that total revenue reaches its highest level in [peak month] and its lowest level in [lowest month]. Darker cells indicate higher revenue, making it easier to identify the strongest sales periods for each category.)*

### Task 4.3 — Correlation heat-map  *(3 pts)*
Compute the Pearson correlation matrix of `quantity, unit_price, discount, customer_age, prep_time_min, rating, revenue`
and plot a **lower-triangle** heat-map (mask the upper triangle) with a **diverging** colour map **centred at 0** (`vmin=-1, vmax=1`), annotated values, square cells.
Name the strongest positive and strongest negative pair (off-diagonal) in a markdown cell.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Select numerical variables
numeric_cols = [
    "quantity", "unit_price", "discount",
    "customer_age", "prep_time_min", "rating", "revenue"
]

# Compute Pearson correlation matrix
corr_matrix = df[numeric_cols].corr(method="pearson")

# Mask the upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

# Plot lower-triangle correlation heat-map
plt.figure(figsize=(10, 8))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson Correlation Coefficient"}
)

plt.title(
    "Pearson Correlation Heat-map of Numerical Variables",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Find strongest positive and negative correlations (off-diagonal)
off_diagonal = corr_matrix.where(
    np.tril(np.ones(corr_matrix.shape, dtype=bool), k=-1)
).stack()

strongest_positive = off_diagonal.idxmax()
strongest_negative = off_diagonal.idxmin()

print(
    f"Strongest positive pair: {strongest_positive[0]} and "
    f"{strongest_positive[1]} "
    f"(r = {off_diagonal.max():.3f})"
)

print(
    f"Strongest negative pair: {strongest_negative[0]} and "
    f"{strongest_negative[1]} "
    f"(r = {off_diagonal.min():.3f})"
)

# Generate the Markdown answer
print("\nMarkdown answer:")
print(
    f"The strongest positive correlation is between "
    f"{strongest_positive[0]} and {strongest_positive[1]} "
    f"(r = {off_diagonal.max():.3f}). "
    f"The strongest negative correlation is between "
    f"{strongest_negative[0]} and {strongest_negative[1]} "
    f"(r = {off_diagonal.min():.3f})."
)

✍️ **Your answer:** *(The strongest positive correlation is between prep_time_min and quantity (r = 0.759), indicating that larger orders tend to require longer preparation times. The strongest negative correlation is between rating and prep_time_min (r = -0.212), suggesting that longer preparation times are weakly associated with lower customer satisfaction ratings.)*

### Task 4.4 — Pair plot  *(2 pts)*
Use `sns.pairplot` on a **random sample of 600 rows** with variables `unit_price, quantity, prep_time_min, revenue`, `hue="category"`, a colour-blind palette, `corner=True`, semi-transparent points and a figure-level title.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Random sample of 600 rows
sample_df = df.sample(n=600, random_state=42)

# Select variables
pair_vars = ["unit_price", "quantity", "prep_time_min", "revenue"]

# Create pair plot
g = sns.pairplot(
    data=sample_df,
    vars=pair_vars,
    hue="category",
    palette="colorblind",
    corner=True,
    plot_kws={"alpha": 0.5, "s": 25},
    diag_kws={"alpha": 0.6}
)

# Add figure-level title
g.figure.suptitle(
    "Relationships Between Price, Quantity, Preparation Time and Revenue",
    fontsize=16,
    fontweight="bold",
    y=1.03
)

plt.show()

### Task 4.5 — Combine Seaborn and Matplotlib customisation  *(2 pts)*
Draw a **Seaborn bar plot** of total revenue per `city` (sorted) and then use **Matplotlib** to:
* write the value (in SAR thousand) above every bar;
* add **one annotation with an arrow** pointing at the best city;
* create a **custom legend** (using `matplotlib.patches.Patch`) that explains *"Best city"* vs *"Other cities"* (colour the best bar differently);
* set custom font sizes for title, labels and ticks.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Calculate total revenue per city and sort descending
city_revenue = (
    df.groupby("city")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

# Identify the best city
best_city = city_revenue.index[0]
best_revenue = city_revenue.iloc[0]

# Colour-blind-safe colours
best_color = sns.color_palette("colorblind")[1]
other_color = sns.color_palette("colorblind")[0]

bar_colors = [
    best_color if city == best_city else other_color
    for city in city_revenue.index
]

# Create Seaborn bar plot
fig, ax = plt.subplots(figsize=(12, 7))

sns.barplot(
    x=city_revenue.index,
    y=city_revenue.values,
    hue=city_revenue.index,
    palette=dict(zip(city_revenue.index, bar_colors)),
    dodge=False,
    legend=False,
    ax=ax
)

# Add revenue labels above each bar (SAR thousands)
for bar in ax.patches:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height + best_revenue * 0.015,
        f"{height / 1000:,.1f}K",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

# Add arrow annotation pointing to the best city
ax.annotate(
    f"Best City: {best_city}",
    xy=(0, best_revenue),
    xytext=(0.8, best_revenue * 1.15),
    fontsize=12,
    fontweight="bold",
    color=best_color,
    arrowprops={
        "arrowstyle": "->",
        "color": best_color,
        "lw": 2
    }
)

# Custom legend using Matplotlib Patch
legend_elements = [
    Patch(facecolor=best_color, label="Best city"),
    Patch(facecolor=other_color, label="Other cities")
]

ax.legend(
    handles=legend_elements,
    title="City Performance",
    fontsize=11,
    title_fontsize=12,
    loc="upper right"
)

# Custom font sizes
ax.set_title(
    "Total Revenue by City",
    fontsize=17,
    fontweight="bold",
    pad=20
)

ax.set_xlabel("City", fontsize=13)
ax.set_ylabel("Total Revenue (SAR)", fontsize=13)
ax.tick_params(axis="x", labelsize=11, rotation=30)
ax.tick_params(axis="y", labelsize=11)

# Leave space for annotations
ax.set_ylim(0, best_revenue * 1.3)

plt.tight_layout()
plt.show()

---
# Part 5 · Exploratory Data Analysis: Correlation & PCA  *(8 pts)*
*Course topic 6: Pearson's r, principal component analysis.*

### Task 5.1 — Is slow service really hurting satisfaction?  *(2 pts)*
Compute **Pearson r** *and* **Spearman ρ** (with p-values) between `prep_time_min` and `rating`. Draw a `sns.regplot` on a 1 500-row sample (add a small `y_jitter` because rating is ordinal).
Explain in 2 sentences which coefficient is more appropriate here and what the result means.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# Compute Pearson and Spearman correlations with p-values
pearson_r, pearson_p = pearsonr(df["prep_time_min"], df["rating"])
spearman_rho, spearman_p = spearmanr(df["prep_time_min"], df["rating"])

print(f"Pearson r = {pearson_r:.3f}, p-value = {pearson_p:.6f}")
print(f"Spearman rho = {spearman_rho:.3f}, p-value = {spearman_p:.6f}")

# Random sample of 1,500 rows
sample_df = df.sample(n=1500, random_state=42)

# Draw regression plot
plt.figure(figsize=(10, 6))

sns.regplot(
    data=sample_df,
    x="prep_time_min",
    y="rating",
    y_jitter=0.15,
    scatter_kws={"alpha": 0.35, "s": 25},
    line_kws={"color": "red", "linewidth": 2},
    color=sns.color_palette("colorblind")[0]
)

plt.title(
    "Relationship Between Preparation Time and Customer Satisfaction",
    fontsize=14,
    fontweight="bold"
)
plt.xlabel("Preparation Time (Minutes)", fontsize=12)
plt.ylabel("Customer Satisfaction Rating (1–5)", fontsize=12)
plt.yticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

# Generate a two-sentence interpretation
strength = "weak" if abs(spearman_rho) < 0.3 else (
    "moderate" if abs(spearman_rho) < 0.7 else "strong"
)
direction = "negative" if spearman_rho < 0 else "positive"
significance = (
    "statistically significant"
    if spearman_p < 0.05
    else "not statistically significant"
)

print("\nMarkdown answer:")
print(
    "Spearman's rho is more appropriate because satisfaction ratings "
    "are ordinal (1–5), while Pearson's r assumes a linear relationship "
    "between numerical variables. "
    f"The results show a {strength} {direction} association "
    f"(Spearman rho = {spearman_rho:.3f}, p = {spearman_p:.4g}) "
    f"that is {significance}, suggesting that longer preparation times "
    "tend to be associated with lower satisfaction ratings."
)

✍️ **Your answer:** *(Spearman's rho is more appropriate because satisfaction ratings are ordinal (1–5), while Pearson's r measures linear relationships between numerical variables. The results show a weak but statistically significant negative association (Spearman ρ = -0.192, p < 0.001), suggesting that longer preparation times tend to be associated with lower customer satisfaction ratings.)*

### Task 5.2 — Principal Component Analysis  *(3 pts)*
1. Standardise `quantity, unit_price, discount, customer_age, prep_time_min, rating, revenue` (`StandardScaler`) — explain in a comment why scaling is necessary.
2. Fit **`pca = PCA()`** keeping **all** components.
3. Plot a **scree plot** (bars of explained-variance ratio) with the **cumulative** variance as a line and a dashed 80 % threshold.
4. Show the **loadings** of the first three PCs as an annotated heat-map (`pd.DataFrame(pca.components_[:3].T, index=features, columns=["PC1","PC2","PC3"])`).

In [ ]:
pca_features = ["quantity", "unit_price", "discount", "customer_age", "prep_time_min", "rating", "revenue"]
pca = None
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Select numerical features
features = [
    "quantity", "unit_price", "discount",
    "customer_age", "prep_time_min", "rating", "revenue"
]

# Standardisation is necessary because PCA is sensitive to feature scales.
# It ensures variables with larger numerical ranges do not dominate the PCA.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

# Fit PCA keeping all components
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
pc_labels = [f"PC{i+1}" for i in range(len(features))]

# Scree plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    pc_labels,
    explained_variance * 100,
    color=sns.color_palette("colorblind")[0],
    alpha=0.8,
    label="Explained Variance (%)"
)

ax.plot(
    pc_labels,
    cumulative_variance * 100,
    color=sns.color_palette("colorblind")[1],
    marker="o",
    linewidth=2,
    label="Cumulative Variance (%)"
)

ax.axhline(
    y=80,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="80% Threshold"
)

ax.set_title(
    "PCA Scree Plot and Cumulative Explained Variance",
    fontsize=15,
    fontweight="bold"
)
ax.set_xlabel("Principal Components", fontsize=12)
ax.set_ylabel("Explained Variance (%)", fontsize=12)
ax.set_ylim(0, 105)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Loadings of the first three principal components
loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=features,
    columns=["PC1", "PC2", "PC3"]
)

# Annotated loadings heat-map
plt.figure(figsize=(8, 6))

sns.heatmap(
    loadings,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"label": "PCA Loading"}
)

plt.title(
    "PCA Loadings: First Three Principal Components",
    fontsize=14,
    fontweight="bold"
)
plt.xlabel("Principal Components")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

# Display explained variance and loadings
print("Explained Variance Ratio:")
display(pd.DataFrame({
    "Component": pc_labels,
    "Explained Variance (%)": (explained_variance * 100).round(2),
    "Cumulative Variance (%)": (cumulative_variance * 100).round(2)
}))

print("\nLoadings of the First Three PCs:")
display(loadings.round(3))

### Task 5.3 — Project and interpret  *(3 pts)*
1. Scatter plot of the data on **PC1 vs PC2** (sample of 2 000 rows, coloured by `category`, axis labels showing the % variance explained).
2. Write 3–4 sentences: how many components do you need for ≥ 80 % variance? Which variables drive PC1 and PC2? Do categories separate in the PC space?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Create PCA projection DataFrame
pca_df = pd.DataFrame(
    X_pca[:, :2],
    columns=["PC1", "PC2"],
    index=df.index
)

pca_df["category"] = df["category"]

# Random sample of 2,000 rows
pca_sample = pca_df.sample(n=2000, random_state=42)

# Scatter plot: PC1 vs PC2
plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=pca_sample,
    x="PC1",
    y="PC2",
    hue="category",
    palette="colorblind",
    alpha=0.55,
    s=35
)

plt.title(
    "PCA Projection of Transactions by Category",
    fontsize=15,
    fontweight="bold"
)

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}% Variance Explained)",
    fontsize=12
)

plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}% Variance Explained)",
    fontsize=12
)

plt.legend(
    title="Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# 2. Number of components needed for at least 80% variance
n_components_80 = np.argmax(
    np.cumsum(pca.explained_variance_ratio_) >= 0.80
) + 1

# Identify variables driving PC1 and PC2
pc1_drivers = loadings["PC1"].abs().nlargest(2).index.tolist()
pc2_drivers = loadings["PC2"].abs().nlargest(2).index.tolist()

print(f"Components needed for >= 80% variance: {n_components_80}")
print(f"Main PC1 drivers: {', '.join(pc1_drivers)}")
print(f"Main PC2 drivers: {', '.join(pc2_drivers)}")

# Generate Markdown answer
print("\nMarkdown answer:")
print(
    f"At least {n_components_80} principal components are needed "
    f"to explain 80% of the total variance. "
    f"PC1 is mainly influenced by {pc1_drivers[0]} and {pc1_drivers[1]}, "
    f"while PC2 is primarily driven by {pc2_drivers[0]} and {pc2_drivers[1]}. "
    "The PCA scatter plot can be used to examine whether product categories "
    "form distinct clusters or overlap in the PC1–PC2 space. "
    "Clear separation would indicate differences between categories, "
    "while substantial overlap would suggest similar numerical characteristics."
)

✍️ **Your answer:** *(Four principal components are needed to explain at least 80% of the total variance. PC1 is mainly driven by revenue and quantity, while PC2 is primarily influenced by unit_price and revenue. The PC1–PC2 scatter plot helps visualize whether product categories form distinct clusters or overlap. The degree of separation between categories can be assessed by examining how much their points overlap in the plot.)*

In [ ]:
# 🔒 CHECK — Part 5 (given)
assert hasattr(pca, "explained_variance_ratio_") and np.isclose(pca.explained_variance_ratio_.sum(), 1.0), "fit PCA() with ALL components"
assert len(pca.explained_variance_ratio_) >= 5
print("✅ Part 5 checks passed")

---
# Part 6 · Time-Series Visualization, Decomposition & Forecasting  *(14 pts)*
*Course topic 7: line/area/bar charts, moving averages, trend & seasonality, stationarity, forecasting.*

### Task 6.1 — Build and plot the daily series  *(3 pts)*
1. Create **`daily`**: total `revenue` per calendar day as a `pd.Series` with a **continuous daily `DatetimeIndex`** (`resample("D").sum()`), named `"revenue"`.
2. Plot it as a thin light line together with the **7-day** and **30-day rolling means** (`rolling().mean()`), with a legend, axis labels and a title.

In [ ]:
daily = None
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Create a continuous daily revenue series
daily = (
    df.set_index("date")["revenue"]
    .resample("D")
    .sum()
)

daily.name = "revenue"

# Calculate rolling means
rolling_7 = daily.rolling(window=7).mean()
rolling_30 = daily.rolling(window=30).mean()

# Colour-blind-safe palette
colors = sns.color_palette("colorblind")

# Plot daily revenue and rolling means
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    daily.index,
    daily.values,
    color="lightgray",
    linewidth=0.8,
    alpha=0.7,
    label="Daily Revenue"
)

ax.plot(
    rolling_7.index,
    rolling_7.values,
    color=colors[0],
    linewidth=2,
    label="7-Day Rolling Mean"
)

ax.plot(
    rolling_30.index,
    rolling_30.values,
    color=colors[1],
    linewidth=2.5,
    label="30-Day Rolling Mean"
)

ax.set_title(
    "Daily Revenue with 7-Day and 30-Day Moving Averages",
    fontsize=15,
    fontweight="bold"
)

ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Revenue (SAR)", fontsize=12)

ax.legend(fontsize=11)
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# 🔒 CHECK — Part 6.1 (given)
assert isinstance(daily, pd.Series) and isinstance(daily.index, pd.DatetimeIndex)
assert len(daily) == 731 and daily.index.freqstr == "D", "daily must cover 2023-01-01 .. 2024-12-31 without gaps"
print("✅ daily series OK:", daily.index.min().date(), "→", daily.index.max().date())

### Task 6.2 — Monthly bar chart and stacked area chart  *(3 pts)*
In a `2 × 1` figure draw (a) a **bar chart** of monthly total revenue (`daily.resample("MS").sum()`), and (b) a **stacked area chart** of monthly revenue by `category`.
Use the same x-range in both panels (`sharex=True`) and format the dates on the x-axis.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Monthly total revenue
monthly_total = daily.resample("MS").sum()

# Monthly revenue by category
monthly_category = (
    df.set_index("date")
    .groupby("category")["revenue"]
    .resample("MS")
    .sum()
    .unstack(level=0, fill_value=0)
)

# Align both monthly series
monthly_category = monthly_category.reindex(
    monthly_total.index, fill_value=0
)

# Colour-blind-safe palette
colors = sns.color_palette(
    "colorblind", n_colors=len(monthly_category.columns)
)

# Create 2 × 1 figure with shared x-axis
fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 10),
    sharex=True,
    constrained_layout=True
)

# (a) Monthly revenue bar chart
axes[0].bar(
    monthly_total.index,
    monthly_total.values,
    width=25,
    color=colors[0],
    alpha=0.85
)

axes[0].set_title(
    "Monthly Total Revenue",
    fontsize=14,
    fontweight="bold"
)
axes[0].set_ylabel("Revenue (SAR)", fontsize=12)
axes[0].grid(axis="y", alpha=0.3)

# (b) Monthly stacked area chart by category
axes[1].stackplot(
    monthly_category.index,
    *[monthly_category[col].values for col in monthly_category.columns],
    labels=monthly_category.columns,
    colors=colors,
    alpha=0.85
)

axes[1].set_title(
    "Monthly Revenue by Category",
    fontsize=14,
    fontweight="bold"
)
axes[1].set_xlabel("Month", fontsize=12)
axes[1].set_ylabel("Revenue (SAR)", fontsize=12)

axes[1].legend(
    title="Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

axes[1].grid(axis="y", alpha=0.3)

# Format dates on shared x-axis
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=1))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

axes[1].tick_params(axis="x", rotation=45)

# Use the same x-range for both panels
axes[1].set_xlim(
    monthly_total.index.min() - pd.Timedelta(days=15),
    monthly_total.index.max() + pd.Timedelta(days=15)
)

fig.suptitle(
    "Brewline Coffee — Monthly Revenue Analysis",
    fontsize=17,
    fontweight="bold"
)

plt.show()

### Task 6.3 — Seasonal decomposition  *(3 pts)*
Apply `statsmodels.tsa.seasonal.seasonal_decompose` (**additive**, `period=7`) to `daily` and plot the four components (observed, trend, seasonal, residual).
Then compute the **average seasonal effect per weekday** (ordered Monday→Sunday) and the **overall trend growth** (2024 vs 2023 total revenue, in %), and interpret them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose

# 1. Additive seasonal decomposition with weekly seasonality
decomposition = seasonal_decompose(
    daily,
    model="additive",
    period=7
)

# Plot the four components
fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)

components = [
    (decomposition.observed, "Observed Revenue"),
    (decomposition.trend, "Trend"),
    (decomposition.seasonal, "Seasonal Component"),
    (decomposition.resid, "Residual")
]

colors = sns.color_palette("colorblind")

for i, (series, title) in enumerate(components):
    axes[i].plot(
        series.index,
        series.values,
        color=colors[i],
        linewidth=1.2
    )
    axes[i].set_title(title, fontsize=12, fontweight="bold")
    axes[i].set_ylabel("Revenue (SAR)")
    axes[i].grid(alpha=0.25)

axes[-1].set_xlabel("Date")

fig.suptitle(
    "Additive Seasonal Decomposition of Daily Revenue (Period = 7)",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# 2. Average seasonal effect by weekday (Monday to Sunday)
weekday_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

seasonal_df = pd.DataFrame({
    "weekday": decomposition.seasonal.index.day_name(),
    "seasonal_effect": decomposition.seasonal.values
})

avg_seasonal_effect = (
    seasonal_df.groupby("weekday")["seasonal_effect"]
    .mean()
    .reindex(weekday_order)
)

print("Average Seasonal Effect by Weekday (SAR):")
display(avg_seasonal_effect.round(2).to_frame("Seasonal Effect (SAR)"))

# Plot weekday seasonal effects
plt.figure(figsize=(11, 5))

bar_colors = [
    colors[1] if day in ["Friday", "Saturday"] else colors[0]
    for day in weekday_order
]

plt.bar(
    weekday_order,
    avg_seasonal_effect.values,
    color=bar_colors
)

plt.axhline(0, color="black", linewidth=1)

plt.title(
    "Average Weekly Seasonal Effect on Revenue",
    fontsize=14,
    fontweight="bold"
)
plt.xlabel("Day of Week")
plt.ylabel("Average Seasonal Effect (SAR)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# 3. Overall revenue growth: 2024 vs 2023
annual_revenue = daily.groupby(daily.index.year).sum()

revenue_2023 = annual_revenue.loc[2023]
revenue_2024 = annual_revenue.loc[2024]

growth_pct = (
    (revenue_2024 - revenue_2023) / revenue_2023
) * 100

print(f"\nTotal Revenue 2023: {revenue_2023:,.2f} SAR")
print(f"Total Revenue 2024: {revenue_2024:,.2f} SAR")
print(f"Revenue Growth (2024 vs 2023): {growth_pct:+.2f}%")

# 4. Generate interpretation
best_day = avg_seasonal_effect.idxmax()
worst_day = avg_seasonal_effect.idxmin()

growth_direction = "increased" if growth_pct > 0 else "decreased"

print("\nMarkdown answer:")
print(
    f"The seasonal decomposition reveals a weekly pattern, with "
    f"{best_day} showing the strongest positive seasonal effect "
    f"({avg_seasonal_effect.max():+.2f} SAR) and "
    f"{worst_day} showing the lowest seasonal effect "
    f"({avg_seasonal_effect.min():+.2f} SAR). "
    f"Total annual revenue {growth_direction} by "
    f"{abs(growth_pct):.2f}% from 2023 to 2024, "
    f"indicating an overall {'upward' if growth_pct > 0 else 'downward'} "
    f"change in sales performance."
)

✍️ **Your answer:** *(The seasonal decomposition reveals a weekly pattern, with Friday showing the strongest positive seasonal effect (+186.69 SAR) and Sunday showing the lowest seasonal effect (-81.70 SAR). This suggests that revenue tends to be higher on Fridays and lower on Sundays compared with the underlying trend. Total annual revenue increased from 131,634.99 SAR in 2023 to 185,788.09 SAR in 2024, representing a 41.14% increase and indicating overall sales growth.)*

### Task 6.4 — Stationarity and differencing  *(2 pts)*
1. Run the **Augmented Dickey-Fuller** test (`statsmodels.tsa.stattools.adfuller`) on `daily` and on its **first difference** (`daily.diff().dropna()`).
2. Plot both series (`2 × 1`) and put the ADF p-value in each title.
3. Explain what a **stationary** series is and which series is stationary at α = 0.05.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

# 1. Augmented Dickey-Fuller test on original daily revenue
adf_original = adfuller(daily.dropna())
p_original = adf_original[1]

# First difference
daily_diff = daily.diff().dropna()

# ADF test on first-differenced series
adf_diff = adfuller(daily_diff)
p_diff = adf_diff[1]

print(f"Original Series ADF p-value: {p_original:.6f}")
print(f"First Difference ADF p-value: {p_diff:.6f}")

# 2. Plot both series
colors = sns.color_palette("colorblind")

fig, axes = plt.subplots(
    2, 1,
    figsize=(14, 9),
    sharex=True,
    constrained_layout=True
)

axes[0].plot(
    daily.index,
    daily.values,
    color=colors[0],
    linewidth=1
)

axes[0].set_title(
    f"Original Daily Revenue (ADF p-value = {p_original:.6f})",
    fontsize=14,
    fontweight="bold"
)
axes[0].set_ylabel("Revenue (SAR)")
axes[0].grid(alpha=0.3)

axes[1].plot(
    daily_diff.index,
    daily_diff.values,
    color=colors[1],
    linewidth=1
)

axes[1].axhline(0, color="black", linestyle="--", linewidth=1)

axes[1].set_title(
    f"First-Differenced Daily Revenue (ADF p-value = {p_diff:.6f})",
    fontsize=14,
    fontweight="bold"
)
axes[1].set_xlabel("Date")
axes[1].set_ylabel("Change in Revenue (SAR)")
axes[1].grid(alpha=0.3)

fig.suptitle(
    "Stationarity Analysis: Original vs First-Differenced Revenue",
    fontsize=16,
    fontweight="bold"
)

plt.show()

# 3. Determine stationarity at alpha = 0.05
alpha = 0.05

original_status = (
    "stationary" if p_original < alpha else "non-stationary"
)

diff_status = (
    "stationary" if p_diff < alpha else "non-stationary"
)

print("\nMarkdown answer:")
print(
    "A stationary time series has statistical properties, such as "
    "its mean, variance, and autocorrelation structure, that remain "
    "approximately constant over time. "
    f"At α = 0.05, the original daily revenue series is {original_status} "
    f"(ADF p = {p_original:.4g}), while the first-differenced series "
    f"is {diff_status} (ADF p = {p_diff:.4g})."
)

✍️ **Your answer:** *(A stationary time series has statistical properties, such as its mean, variance, and autocorrelation structure, that remain approximately constant over time. At α = 0.05, the original daily revenue series is non-stationary (ADF p = 0.3286 > 0.05). However, the first-differenced series is stationary (ADF p < 0.001), indicating that first-order differencing successfully removes the non-stationarity detected by the ADF test.)*

### Task 6.5 — Forecast the next 28 days  *(3 pts)*
1. Hold out the **last 28 days** as a test set (`train`, `test`).
2. Fit a **Holt-Winters** model (`ExponentialSmoothing`, additive trend, additive seasonality, `seasonal_periods=7`) on `train` and store the 28-day forecast in **`forecast_hw`** (a Series with the test index).
3. Build a **seasonal-naive baseline** `forecast_naive`: repeat the last 7 observed days four times (same test index).
4. Create `metrics`: a DataFrame with index `["Holt-Winters", "Seasonal naive"]` and columns `["MAE", "MAPE (%)"]` computed on the test set.
5. Plot the last 60 training days, the actual test values and both forecasts; mention the better model and its error in the title or a text note.

In [ ]:
H = 28
forecast_hw = None
forecast_naive = None
metrics = None
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error

# 1. Split data: last 28 days for testing
train = daily.iloc[:-28]
test = daily.iloc[-28:]

# 2. Fit Holt-Winters model
hw_model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=7,
    initialization_method="estimated"
).fit()

# Forecast the next 28 days
forecast_hw = pd.Series(
    hw_model.forecast(28).to_numpy(),
    index=test.index,
    name="Holt-Winters"
)

# 3. Seasonal-naive forecast: repeat last 7 observed days
forecast_naive = pd.Series(
    np.tile(train.iloc[-7:].to_numpy(), 4),
    index=test.index,
    name="Seasonal naive"
)

# 4. Calculate MAE and MAPE
def calculate_mape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    # Exclude zero actual values to avoid division by zero
    mask = actual != 0

    if not mask.any():
        return np.nan

    return np.mean(
        np.abs((actual[mask] - predicted[mask]) / actual[mask])
    ) * 100

metrics = pd.DataFrame(
    {
        "MAE": [
            mean_absolute_error(test, forecast_hw),
            mean_absolute_error(test, forecast_naive)
        ],
        "MAPE (%)": [
            calculate_mape(test, forecast_hw),
            calculate_mape(test, forecast_naive)
        ]
    },
    index=["Holt-Winters", "Seasonal naive"]
)

print("Forecast Evaluation Metrics:")
display(metrics.round(2))

# Identify the better model based on MAE
best_model = metrics["MAE"].idxmin()
best_mae = metrics.loc[best_model, "MAE"]
best_mape = metrics.loc[best_model, "MAPE (%)"]

# 5. Plot last 60 training days, test data and forecasts
colors = sns.color_palette("colorblind")

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    train.iloc[-60:].index,
    train.iloc[-60:].values,
    label="Training Data (Last 60 Days)",
    color=colors[0],
    linewidth=1.5
)

ax.plot(
    test.index,
    test.values,
    label="Actual Test Data",
    color="black",
    linewidth=2,
    marker="o",
    markersize=3
)

ax.plot(
    forecast_hw.index,
    forecast_hw.values,
    label="Holt-Winters Forecast",
    color=colors[1],
    linewidth=2,
    linestyle="--"
)

ax.plot(
    forecast_naive.index,
    forecast_naive.values,
    label="Seasonal-Naive Forecast",
    color=colors[2],
    linewidth=2,
    linestyle=":"
)

ax.axvline(
    test.index[0],
    color="gray",
    linestyle="--",
    alpha=0.7,
    label="Train/Test Split"
)

ax.set_title(
    f"28-Day Revenue Forecast | Best Model: {best_model} "
    f"(MAE = {best_mae:,.2f} SAR)",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Date", fontsize=12)
ax.set_ylabel("Daily Revenue (SAR)", fontsize=12)

ax.legend(fontsize=10)
ax.grid(alpha=0.25)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Generate interpretation
print("\nMarkdown answer:")
print(
    f"The {best_model} model achieved the lowest MAE "
    f"({best_mae:,.2f} SAR) and is therefore the better model "
    f"based on this evaluation metric. "
    f"Its MAPE is {best_mape:.2f}%, indicating the average "
    f"absolute percentage forecasting error on the test set."
)

In [ ]:
# 🔒 CHECK — Part 6.5 (given)
assert len(forecast_hw) == len(forecast_naive) == 28
assert list(metrics.index) == ["Holt-Winters", "Seasonal naive"] and list(metrics.columns) == ["MAE", "MAPE (%)"]
assert (metrics > 0).all().all()
print("✅ Part 6 checks passed")

---
# Part 7 · Interactive Visualization with Plotly  *(10 pts)*
*Course topic 5: Plotly Express, graph objects, animation, subplots, hover, range sliders.*
All figures must have a title, axis titles and informative **hover** text. Show them with `fig.show()`.

### Task 7.1 — Interactive bubble chart  *(2 pts)*
Aggregate by `category` and `product` (average price, total units, total revenue, average rating) and create a `px.scatter`: x = average price, y = total units, **bubble size = revenue**, colour = category, `hover_name` = product, extra `hover_data` (avg rating, revenue formatted).

In [ ]:
import pandas as pd
import plotly.express as px

# Aggregate data by category and product
product_summary = (
    df.groupby(["category", "product"], as_index=False)
    .agg(
        avg_price=("unit_price", "mean"),
        total_units=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_rating=("rating", "mean")
    )
)

# Create interactive bubble chart
fig = px.scatter(
    product_summary,
    x="avg_price",
    y="total_units",
    size="total_revenue",
    color="category",
    hover_name="product",
    hover_data={
        "category": True,
        "avg_price": ":.2f",
        "total_units": ":,",
        "total_revenue": ":,.2f",
        "avg_rating": ":.2f"
    },
    size_max=60,
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Product Performance: Average Price, Units Sold and Revenue",
    labels={
        "avg_price": "Average Unit Price (SAR)",
        "total_units": "Total Units Sold",
        "total_revenue": "Total Revenue (SAR)",
        "avg_rating": "Average Satisfaction Rating",
        "category": "Product Category"
    }
)

# Customize layout
fig.update_layout(
    title_font_size=18,
    xaxis_title="Average Unit Price (SAR)",
    yaxis_title="Total Units Sold",
    legend_title="Category",
    template="plotly_white",
    height=650
)

# Display interactive chart
fig.show()

### Task 7.2 — Animated bar chart  *(3 pts)*
Create an **animated** `px.bar` of monthly revenue per category: x = `category`, y = revenue, `animation_frame="year_month"`, colour = category.
Fix the y-axis range to the global maximum (so bars are comparable between frames) and make sure the **play button and slider** are visible.

In [ ]:
import pandas as pd
import plotly.express as px

# Aggregate monthly revenue by category
monthly_revenue = (
    df.groupby(["year_month", "category"], as_index=False)
    .agg(revenue=("revenue", "sum"))
)

# Ensure all categories appear in every month
months = sorted(monthly_revenue["year_month"].unique())
categories = sorted(monthly_revenue["category"].unique())

full_index = pd.MultiIndex.from_product(
    [months, categories],
    names=["year_month", "category"]
)

monthly_revenue = (
    monthly_revenue
    .set_index(["year_month", "category"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

monthly_revenue["year_month"] = monthly_revenue["year_month"].astype(str)

# Global maximum for a fixed y-axis range
max_revenue = monthly_revenue["revenue"].max()

# Create animated bar chart
fig = px.bar(
    monthly_revenue,
    x="category",
    y="revenue",
    color="category",
    animation_frame="year_month",
    category_orders={
        "year_month": [str(m) for m in months],
        "category": categories
    },
    range_y=[0, max_revenue * 1.15],
    color_discrete_sequence=px.colors.qualitative.Safe,
    hover_data={
        "year_month": True,
        "revenue": ":,.2f"
    },
    labels={
        "category": "Product Category",
        "revenue": "Monthly Revenue (SAR)",
        "year_month": "Month"
    },
    title="Animated Monthly Revenue by Product Category"
)

# Customize chart
fig.update_layout(
    template="plotly_white",
    title_font_size=18,
    xaxis_title="Product Category",
    yaxis_title="Total Revenue (SAR)",
    legend_title="Category",
    height=650,
    margin=dict(b=140)
)

# Keep the play button and slider visible
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            showactive=False,
            x=0.05,
            y=-0.08,
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        {
                            "frame": {
                                "duration": 700,
                                "redraw": True
                            },
                            "transition": {
                                "duration": 300
                            },
                            "fromcurrent": True
                        }
                    ]
                ),
                dict(
                    label="❚❚ Pause",
                    method="animate",
                    args=[
                        [None],
                        {
                            "frame": {
                                "duration": 0,
                                "redraw": False
                            },
                            "mode": "immediate"
                        }
                    ]
                )
            ]
        )
    ]
)

# Adjust animation slider position
if fig.layout.sliders:
    fig.layout.sliders[0].update(
        visible=True,
        y=-0.08,
        x=0.25,
        len=0.7
    )

fig.show()

### Task 7.3 — Dual-axis chart  *(2 pts)*
With `plotly.subplots.make_subplots(specs=[[{"secondary_y": True}]])` plot **monthly revenue as bars** (primary axis) and the **monthly average rating as a line with markers** (secondary axis, range 3–5).
Label both y-axes, use a unified hover (`hovermode="x unified"`).

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Aggregate monthly revenue and average rating
monthly_summary = (
    df.groupby("year_month")
    .agg(
        total_revenue=("revenue", "sum"),
        avg_rating=("rating", "mean")
    )
    .sort_index()
    .reset_index()
)

# Convert year_month to datetime
monthly_summary["year_month"] = pd.to_datetime(
    monthly_summary["year_month"].astype(str)
)

# Create dual-axis chart
fig = make_subplots(
    specs=[[{"secondary_y": True}]]
)

# Monthly revenue bars (primary y-axis)
fig.add_trace(
    go.Bar(
        x=monthly_summary["year_month"],
        y=monthly_summary["total_revenue"],
        name="Monthly Revenue",
        marker_color="#0072B2",
        hovertemplate="Revenue: %{y:,.2f} SAR<extra></extra>"
    ),
    secondary_y=False
)

# Monthly average rating line (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=monthly_summary["year_month"],
        y=monthly_summary["avg_rating"],
        mode="lines+markers",
        name="Average Rating",
        line=dict(color="#D55E00", width=3),
        marker=dict(size=8),
        hovertemplate="Average Rating: %{y:.2f}/5<extra></extra>"
    ),
    secondary_y=True
)

# Chart title and layout
fig.update_layout(
    title="Monthly Revenue and Average Customer Satisfaction",
    title_font_size=18,
    xaxis_title="Month",
    hovermode="x unified",
    template="plotly_white",
    height=600,
    legend=dict(
        orientation="h",
        y=1.12,
        x=0.5,
        xanchor="center"
    )
)

# Primary y-axis
fig.update_yaxes(
    title_text="Total Revenue (SAR)",
    secondary_y=False
)

# Secondary y-axis
fig.update_yaxes(
    title_text="Average Rating (1–5)",
    range=[3, 5],
    secondary_y=True
)

# Format dates
fig.update_xaxes(
    tickformat="%b %Y",
    dtick="M1",
    tickangle=-45
)

# Display interactive chart
fig.show()

### Task 7.4 — Geographic bubble map  *(2 pts)*
Compute total revenue per store (`store_id`, `city`, `store_size`, `lat`, `lon`) and draw a `px.scatter_geo` (or `px.scatter_map`) with bubble size = revenue, colour = store size and hover showing store id, city and revenue. Zoom to the stores (`fitbounds="locations"`).

In [ ]:
import pandas as pd
import plotly.express as px

# Aggregate total revenue per store
store_revenue = (
    df.groupby(
        ["store_id", "city", "store_size", "lat", "lon"],
        as_index=False
    )
    .agg(total_revenue=("revenue", "sum"))
)

# Create interactive geographic bubble map
fig = px.scatter_geo(
    store_revenue,
    lat="lat",
    lon="lon",
    size="total_revenue",
    color="store_size",
    hover_name="store_id",
    hover_data={
        "city": True,
        "store_size": True,
        "total_revenue": ":,.2f",
        "lat": False,
        "lon": False
    },
    size_max=45,
    fitbounds="locations",
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Geographic Distribution of Store Revenue",
    labels={
        "store_id": "Store ID",
        "city": "City",
        "store_size": "Store Size",
        "total_revenue": "Total Revenue (SAR)"
    }
)

# Customize map layout
fig.update_geos(
    visible=True,
    showland=True,
    landcolor="rgb(245, 245, 245)",
    showcountries=True,
    countrycolor="gray",
    showcoastlines=True
)

fig.update_layout(
    title_font_size=18,
    legend_title="Store Size",
    template="plotly_white",
    height=650,
    margin=dict(l=20, r=20, t=80, b=20)
)

# Display interactive map
fig.show()

### Task 7.5 — Range slider and standalone HTML  *(1 pt)*
Plot `daily` revenue as an interactive line (`px.line`) with a **range slider** and **range-selector buttons** (1 m, 6 m, all), then export it with `fig.write_html("outputs/daily_revenue_interactive.html", include_plotlyjs="cdn")`.

In [ ]:
import os
import pandas as pd
import plotly.express as px

# Prepare daily revenue data
daily_df = daily.reset_index()
daily_df.columns = ["date", "revenue"]

# Create interactive line chart
fig = px.line(
    daily_df,
    x="date",
    y="revenue",
    title="Interactive Daily Revenue Over Time",
    labels={
        "date": "Date",
        "revenue": "Daily Revenue (SAR)"
    },
    hover_data={
        "date": "|%d %b %Y",
        "revenue": ":,.2f"
    }
)

# Add range slider and range-selector buttons
fig.update_xaxes(
    title_text="Date",
    rangeslider=dict(visible=True),
    rangeselector=dict(
        buttons=[
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(step="all", label="All")
        ]
    )
)

# Customize layout
fig.update_layout(
    template="plotly_white",
    title_font_size=18,
    yaxis_title="Daily Revenue (SAR)",
    hovermode="x unified",
    height=650,
    margin=dict(t=100, b=80)
)

# Display interactive chart
fig.show()

# Export as standalone HTML
os.makedirs("outputs", exist_ok=True)

fig.write_html(
    "outputs/daily_revenue_interactive.html",
    include_plotlyjs="cdn"
)

print("HTML saved successfully: outputs/daily_revenue_interactive.html")

---
# Part 8 · Data Storytelling & Reflection  *(10 pts)*
*All topics — communicating insights effectively.*

### Task 8.1 — One story figure  *(4 pts)*
Design **one publication-quality figure** that communicates the **single most important insight** you found, for a non-technical manager. It must have:
* a **headline title that states the finding** (not just "Revenue by month"), and a subtitle/source note;
* **direct annotations** or highlighted regions instead of heavy gridlines/legends;
* a colour-blind-safe palette, minimal chart junk, readable fonts (≥ 11 pt);
* saved as `outputs/story_figure.png` (300 dpi).

*(Suggestion used in the reference solution: share of monthly revenue of **Cold Drinks vs Coffee** with summer months shaded — but you may choose any insight you can defend.)*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Task 8.1 — One Story Figure

# 1. Prepare monthly revenue data for Cold Drinks and Coffee
story_df = df.copy()
story_df["date"] = pd.to_datetime(story_df["date"])

story_df = story_df[
    story_df["category"].isin(["Cold Drinks", "Coffee"])
].copy()

story_df["month_start"] = story_df["date"].dt.to_period("M").dt.to_timestamp()

monthly_story = (
    story_df.groupby(["month_start", "category"])["revenue"]
    .sum()
    .unstack(fill_value=0)
    .sort_index()
)

monthly_story = monthly_story.reindex(
    columns=["Cold Drinks", "Coffee"],
    fill_value=0
)

# 2. Calculate monthly revenue shares
monthly_story["total"] = (
    monthly_story["Cold Drinks"] + monthly_story["Coffee"]
)

monthly_story["cold_share"] = (
    monthly_story["Cold Drinks"] / monthly_story["total"] * 100
)

monthly_story["coffee_share"] = (
    monthly_story["Coffee"] / monthly_story["total"] * 100
)

# 3. Compare summer months with other months
summer_mask = monthly_story.index.month.isin([6, 7, 8])

summer_cold_share = (
    monthly_story.loc[summer_mask, "Cold Drinks"].sum()
    / monthly_story.loc[summer_mask, "total"].sum()
    * 100
)

other_cold_share = (
    monthly_story.loc[~summer_mask, "Cold Drinks"].sum()
    / monthly_story.loc[~summer_mask, "total"].sum()
    * 100
)

difference = summer_cold_share - other_cold_share

# 4. Create a headline that states the actual finding
if difference > 0:
    headline = (
        f"Cold Drinks Gain {difference:.1f} Percentage Points "
        "of Revenue Share in Summer"
    )
elif difference < 0:
    headline = (
        f"Cold Drinks Lose {abs(difference):.1f} Percentage Points "
        "of Revenue Share in Summer"
    )
else:
    headline = "Cold Drinks Maintain Their Revenue Share in Summer"

# 5. Set publication-quality formatting
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11
})

fig, ax = plt.subplots(figsize=(15, 7.5))

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

# Colour-blind-safe palette
cold_color = "#0072B2"
coffee_color = "#D55E00"
summer_color = "#F0E442"

# 6. Shade summer months (June–August)
for year in sorted(monthly_story.index.year.unique()):
    ax.axvspan(
        pd.Timestamp(year, 6, 1),
        pd.Timestamp(year, 9, 1),
        color=summer_color,
        alpha=0.18,
        zorder=0
    )

# 7. Plot revenue shares
ax.plot(
    monthly_story.index,
    monthly_story["cold_share"],
    color=cold_color,
    linewidth=3,
    marker="o",
    markersize=6,
    zorder=3
)

ax.plot(
    monthly_story.index,
    monthly_story["coffee_share"],
    color=coffee_color,
    linewidth=3,
    marker="o",
    markersize=6,
    zorder=3
)

# 8. Add direct annotations instead of a legend
last_date = monthly_story.index[-1]

ax.annotate(
    f"Cold Drinks: {monthly_story['cold_share'].iloc[-1]:.1f}%",
    xy=(last_date, monthly_story["cold_share"].iloc[-1]),
    xytext=(12, 0),
    textcoords="offset points",
    color=cold_color,
    fontsize=12,
    fontweight="bold",
    va="center"
)

ax.annotate(
    f"Coffee: {monthly_story['coffee_share'].iloc[-1]:.1f}%",
    xy=(last_date, monthly_story["coffee_share"].iloc[-1]),
    xytext=(12, 0),
    textcoords="offset points",
    color=coffee_color,
    fontsize=12,
    fontweight="bold",
    va="center"
)

# 9. Add headline and subtitle
fig.suptitle(
    headline,
    fontsize=18,
    fontweight="bold",
    x=0.08,
    y=0.98,
    ha="left"
)

fig.text(
    0.08,
    0.92,
    "Monthly Revenue Share: Cold Drinks vs Coffee | "
    "Summer Months Highlighted (June–August)",
    fontsize=12,
    color="#555555",
    ha="left"
)

# 10. Highlight the key finding
summary_text = (
    f"Summer Cold Drinks Share: {summer_cold_share:.1f}%\n"
    f"Other Months Share: {other_cold_share:.1f}%\n"
    f"Difference: {difference:+.1f} percentage points"
)

ax.text(
    0.02,
    0.97,
    summary_text,
    transform=ax.transAxes,
    fontsize=12,
    fontweight="bold",
    color=cold_color,
    va="top",
    ha="left",
    linespacing=1.5,
    bbox=dict(
        facecolor="white",
        edgecolor="#DDDDDD",
        boxstyle="round,pad=0.6",
        alpha=0.95
    )
)

# 11. Format axes
ax.set_xlabel("Month")
ax.set_ylabel("Share of Combined Revenue (%)")
ax.set_ylim(0, 100)

ax.set_xlim(
    monthly_story.index.min() - pd.Timedelta(days=10),
    monthly_story.index.max() + pd.Timedelta(days=65)
)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

plt.setp(ax.get_xticklabels(), rotation=35, ha="right")

# Remove unnecessary chart elements
ax.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.spines["bottom"].set_color("#BBBBBB")
ax.tick_params(axis="both", length=0, pad=8)

# 12. Add source note
fig.text(
    0.08,
    0.035,
    "Source: Brewline cleaned transaction data (2023–2024). "
    "Shaded areas represent summer months. "
    "Revenue shares are calculated within Cold Drinks and Coffee only.",
    fontsize=11,
    color="#666666",
    ha="left"
)

fig.subplots_adjust(
    left=0.08,
    right=0.84,
    top=0.84,
    bottom=0.17
)

# 13. Display and save figure
os.makedirs("outputs", exist_ok=True)

plt.show()

fig.savefig("outputs/story_figure.png", dpi=300, bbox_inches="tight")

### Task 8.2 — Three insights and two recommendations  *(4 pts)*
In the Markdown cell below write **three insights** (each supported by a **specific number from your analysis** and a reference to the figure/task where it appears) and **two concrete recommendations** for BrewLine management.
Insights must be *findings*, not descriptions ("Riyadh earns 41 % of revenue" ✔, "I made a bar chart" ✘).

✍️ **Your insights and recommendations**

**Insight 1:** Strong Revenue Growth (Task 6.3: Time-Series Analysis)

BrewLine's annual revenue increased from 131,634.99 SAR in 2023 to 185,788.09 SAR in 2024, representing a 41.14% increase. This indicates strong overall business growth.

**Insight 2:** Friday Is the Strongest Sales Day (Task 6.3: Seasonal Decomposition)

The seasonal decomposition shows that Friday has the highest positive seasonal effect (+186.69 SAR), while Sunday has the lowest (-81.70 SAR). This indicates stronger customer demand on Fridays and weaker demand on Sundays.

**Insight 3:** Longer Preparation Times Are Associated with Lower Ratings (Tasks 4.3 and 5.1: Correlation Analysis)

Preparation time and customer satisfaction have a negative Spearman correlation of -0.192 (p < 0.001). This suggests that customers tend to give lower satisfaction ratings when their orders take longer to prepare.

**Recommendation 1:** Optimize Staffing and Inventory for Peak Demand

BrewLine should increase staff coverage and prepare sufficient inventory on Fridays to handle higher customer demand. Management should also consider targeted Sunday promotions to encourage sales during weaker periods.

**Recommendation 2:** Reduce Preparation Times to Improve Customer Satisfaction

BrewLine should monitor preparation times, streamline order preparation workflows, and provide additional staff support during busy hours. Reducing customer waiting times may help improve satisfaction ratings and encourage repeat purchases.

### Task 8.3 — Reflection  *(2 pts)*
In **4–6 sentences** answer: (a) Which **two charts** from this project were the most effective and why? (b) Name **one chart you would *not* use again** (or one design mistake you corrected). (c) What are **two limitations** of the data/analysis (e.g. simulated data, outlier removal choices, imputation, forecast horizon) and how could they mislead a decision-maker?

✍️ **Your reflection:** *( Task 8.3 — Reflection

The two most effective charts were the **seasonal decomposition chart (Task 6.3)** and the **storytelling figure (Task 8.1)** because they clearly highlighted weekly revenue patterns and the differences in revenue share between Cold Drinks and Coffee. The seasonal decomposition chart helped identify peak sales days, while the storytelling figure used direct annotations and highlighted summer months to communicate insights to non-technical managers. I would not use the **PCA pairwise visualization (Task 4.4)** again for management presentations because its multiple plots can be difficult for non-technical audiences to interpret. One limitation is that the dataset may be simulated, meaning the observed patterns might not fully represent real customer behavior and could lead to unrealistic business expectations. Another limitation is that removing outliers may exclude genuine high-value transactions, potentially underestimating revenue and influencing management decisions.)*

---
## ✅ Final submission checklist
- [ ] Name and ID filled in at the top
- [ ] *Kernel → Restart & Run All* finishes without errors and no `YOUR CODE HERE` stubs remain
- [ ] All 🔒 check cells print ✅
- [ ] Every figure has title, axis labels, readable fonts and an accessible palette
- [ ] Every "interpret / explain" question is answered **with numbers**
- [ ] `outputs/` contains: `brewline_clean.csv`, `fig_3_categorical.png`, `daily_revenue_interactive.html`, `story_figure.png`
- [ ] Notebook (with outputs) + `outputs/` folder zipped and submitted

*Good luck — and remember: a chart is only as good as the decision it helps someone make.* ☕

In [ ]:
# 🔒 FINAL SELF-CHECK (given)
required = ["outputs/brewline_clean.csv", "outputs/fig_3_categorical.png", "outputs/daily_revenue_interactive.html", "outputs/story_figure.png"]
for f in required:
    print(("✅" if os.path.exists(f) else "❌ MISSING"), f)
print("Student:", STUDENT_NAME, STUDENT_ID)